# Lenguajes y Autómatas II (SCD-1016)
## Sesión 01: Encuadre, Diagnóstico de Punteros y Metodología IA
**Docente Titular:** MCC. Iván Márquez Larios (`ijmarquezl`)
**Repositorio Maestro:** `https://github.com/ijmarquezl/tecnm-lenguajes-automatas-2`

---
### Protocolo de la Sesión
1. **Fase 1 (El Artesano):** Resolver a mano: *Inversión de buffer in-place (char*)*.
2. **Fase 2 (El Científico):** Definir contrato C-R-E-O: *Punteros, O(n) tiempo, O(1) memoria* y formular prompt.
3. **Fase 3 (El Auditor):** Someter la solución al Test Harness: *Valores NULL, buffers vacios y longitudes impares*.
4. **Entrega:** Documentar en `auditorias/auditoria_sesion01.md`.

### Paso 0: Inicialización del Entorno (C y Rust)

In [1]:
import os
if os.system('which rustc > /dev/null 2>&1') != 0:
    print('⏳ Instalando Rust en Colab...')
    !apt-get update -qq && apt-get install -y -qq rustc > /dev/null 2>&1
print('✅ Entorno listo:')
!gcc --version | head -n 1
!rustc --version

⏳ Instalando Rust en Colab...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Entorno listo:
gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0
rustc 1.75.0 (82e1608df 2023-12-21) (built from a source tarball)


---
## SECCIÓN A: TRACK C
### Fase 1: El Artesano (15 min - Sin IA)
**Reto:** Inversión de buffer in-place (char*)

In [2]:
%%writefile artesano_sesion01_c.c
#include <stdio.h>
#include <string.h>

void invertir(char *texto) {
    if (texto == NULL) {
        return;
    }

    char *inicio = texto;
    char *fin = texto + strlen(texto);

    if (inicio == fin) {
        return;
    }

    fin--;

    while (inicio < fin) {
        char temporal = *inicio;
        *inicio = *fin;
        *fin = temporal;

        inicio++;
        fin--;
    }
}

int main(void) {
    char palabra[] = "HOLA";

    printf("Palabra original: %s\n", palabra);

    invertir(palabra);

    printf("Palabra invertida: %s\n", palabra);

    return 0;
}




Writing artesano_sesion01_c.c


In [3]:
!gcc -Wall -Wextra artesano_sesion01_c.c -o bin_c && ./bin_c

Palabra original: HOLA
Palabra invertida: ALOH


### Fase 2: El Científico (Prompting C-R-E-O)
Pega aquí la implementación optimizada propuesta por la IA:

In [4]:
%%writefile ia_sesion01_c.c
#include <stdio.h>
#include <string.h>

void invertir(char *texto) {
    if (texto == NULL) {
        return;
    }

    size_t longitud = strlen(texto);

    for (size_t i = 0; i < longitud / 2; i++) {
        char temporal = texto[i];
        texto[i] = texto[longitud - i - 1];
        texto[longitud - i - 1] = temporal;
    }
}

int main(void) {
    char palabra[] = "MUNDO";

    printf("Original: %s\n", palabra);

    invertir(palabra);

    printf("Invertida: %s\n", palabra);

    return 0;
}

// Pega aquí el código generado por la IA para auditarlo


Writing ia_sesion01_c.c


In [5]:
!gcc -Wall -Wextra ia_sesion01_c.c -o bin_ia && ./bin_ia

Original: MUNDO
Invertida: ODNUM


### Fase 3: El Auditor (Test Harness en C)
Pruebas de estrés: Valores NULL, buffers vacios y longitudes impares

In [6]:
%%writefile harness_sesion01_c.c

#include <stdio.h>
#include <string.h>
#include <assert.h>

void invertir(char *texto) {
    if (texto == NULL) {
        return;
    }

    size_t longitud = strlen(texto);

    for (size_t i = 0; i < longitud / 2; i++) {
        char temporal = texto[i];
        texto[i] = texto[longitud - i - 1];
        texto[longitud - i - 1] = temporal;
    }
}

int main(void) {
    printf("=== AUDITORIA CRITICA: SESION 01 ===\n");

    // Prueba 1: cadena normal
    char texto1[] = "HOLA";
    invertir(texto1);
    assert(strcmp(texto1, "ALOH") == 0);
    printf("Prueba 1: Correcta\n");

    // Prueba 2: longitud impar
    char texto2[] = "ABCDE";
    invertir(texto2);
    assert(strcmp(texto2, "EDCBA") == 0);
    printf("Prueba 2: Correcta\n");

    // Prueba 3: cadena vacia
    char texto3[] = "";
    invertir(texto3);
    assert(strcmp(texto3, "") == 0);
    printf("Prueba 3: Correcta\n");

    // Prueba 4: valor NULL
    invertir(NULL);
    printf("Prueba 4: Correcta\n");

    // Prueba 5: un caracter
    char texto4[] = "A";
    invertir(texto4);
    assert(strcmp(texto4, "A") == 0);
    printf("Prueba 5: Correcta\n");

    printf("TODAS LAS PRUEBAS PASARON\n");

    return 0;
}



Writing harness_sesion01_c.c


In [8]:
!gcc -Wall -Wextra harness_sesion01_c.c -o test_bin && ./test_bin

=== AUDITORIA CRITICA: SESION 01 ===
Prueba 1: Correcta
Prueba 2: Correcta
Prueba 3: Correcta
Prueba 4: Correcta
Prueba 5: Correcta
TODAS LAS PRUEBAS PASARON
